# 04 — Muestra de unión para la base de elección modal

Genera una muestra local de 350 viajes en total, proporcional por modo y reproducible, para inspeccionar cómo se relacionan `viajes`, `personas`, `hogares`, `vehiculos`, `etapas` y `ZAT_Distance.csv`. Se mantiene una fila por viaje: persona y hogar se unen por las llaves validadas; vehículos y etapas se resumen para la demostración porque son relaciones uno a muchos.

`zat_centroid_distance_m` usa `NEAR_DIST` en metros; los códigos ZAT enteros se convierten solo para comparar su representación numérica, y los intrazonales llevan cero derivado con estado explícito. Las sumas de tiempos/costos y las secuencias de modos de etapa son exploratorias, no reglas de agregación aprobadas. El CSV contiene microdatos e identificadores y permanece bajo `data/interim/`, excluido de Git. No representa una muestra para inferencia.

In [ ]:
from pathlib import Path
import pandas as pd
from mode_choice.mode_choice_join_preview import build_join_preview

PROJECT_ROOT = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (candidate / 'pyproject.toml').is_file()), None)
assert PROJECT_ROOT is not None, 'Could not locate the project root.'
preview_path, summary = build_join_preview(PROJECT_ROOT, sample_size=350, random_seed=5082026)
preview = pd.read_csv(preview_path, dtype=str, keep_default_na=False)
assert len(preview) == summary['preview_rows'] == 350
assert preview['key_viaje'].nunique() == len(preview)
assert preview['persona__key_persona'].ne('').all()
assert preview['hogar__key_hg'].ne('').all()
assert sum(summary['sample_mode_counts'].values()) == 350
assert summary['distance_table_validation'] == {'row_count': 1475010, 'unique_directed_pairs': 1475010, 'zone_count': 1215}
assert summary['distance_status_counts'].get('unmatched_zone_pair', 0) == 0
assert summary['distance_status_counts'].get('missing_zone_code', 0) == 0
assert (preview.loc[preview.zat_distance_status == 'intrazonal_zero_derived', 'zat_centroid_distance_m'] == '0').all()
summary

In [ ]:
print({
    'preview_rows': len(preview),
    'sample_mode_counts': summary['sample_mode_counts'],
    'distance_status_counts': summary['distance_status_counts'],
    'available_column_count': len(preview.columns),
    'local_preview_file': preview_path.relative_to(PROJECT_ROOT).as_posix(),
})

## Recomendación de estructura

Para la futura base plana, conservar una fila por `key_viaje`; unir persona por `key_pers = key_persona` y hogar por `key_hg`. No unir directamente filas de `etapas` o `vehiculos`: eso repetiría el viaje. Mantener las tablas de etapas y vehículos como tablas relacionadas o aprobar explícitamente agregados por viaje/hogar con definiciones documentadas. Elegir variables duplicadas solo después de revisar `outputs/tables/mode_choice_duplicate_field_comparison.csv`. El modo observado es la alternativa elegida; no equivale a contar con atributos de opciones no elegidas.